# imgbeddings using CLIP CPU

https://github.com/albertferre/shelf-product-identifier
https://github.com/MathMagicx/JupyterNotebooks/blob/master/ImageRecommenderResnet18/Recommending%20Similar%20Images.ipynb

In [1]:
from PIL import Image
from imgbeddings import imgbeddings
from os import path

/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
img_path = '/mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0393.jpg'
img_path2 = '/mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0405.jpg'

In [3]:
image = Image.open(img_path)
image2 = Image.open(img_path2)

In [4]:
ibed = imgbeddings()

/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/huggingface_hub/file_download.py:659: FutureWarning: 'cached_download' is the legacy way to download files from the HF hub, please consider upgrading to 'hf_hub_download'
  warnings.warn(
/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/transformers/models/clip/processing_clip.py:143: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(


In [6]:
embedding = ibed.to_embeddings([image, image2, image, image2])
# 2.5-3 seconds for 4 images

In [8]:
embedding

array([[ 0.3170503 ,  0.4797657 , -0.6822478 , ...,  0.5263298 ,
         0.5417492 ,  0.71916586],
       [ 0.2399538 ,  0.2259715 , -0.46870315, ...,  0.72493005,
         0.8082715 ,  0.7703428 ],
       [ 0.3170503 ,  0.4797657 , -0.6822478 , ...,  0.5263298 ,
         0.5417492 ,  0.71916586],
       [ 0.2399538 ,  0.2259715 , -0.46870315, ...,  0.72493005,
         0.8082715 ,  0.7703428 ]], dtype=float32)

In [10]:
len(embedding[0])
# embedding length

768

# Media Pipe Embeddings

In [11]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [12]:
model_path = 'mobilenet_v3_small_075_224_embedder.tflite'

In [14]:
BaseOptions = mp.tasks.BaseOptions
ImageEmbedder = mp.tasks.vision.ImageEmbedder
ImageEmbedderOptions = mp.tasks.vision.ImageEmbedderOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = ImageEmbedderOptions(
    base_options=BaseOptions(),
    quantize=True,
    running_mode=VisionRunningMode.IMAGE)

with ImageEmbedder.create_from_options(options) as embedder:
    # The embedder is initialized. Use it here.
    # Load the input image from an image file.
    mp_image = mp.Image.create_from_file(img_path)
    mp_image2 = mp.Image.create_from_file(img_path2)
    # Perform image embedding on the provided single image.
    embedding_result = embedder.embed(mp_image)
    embedding_result2 = embedder.embed(mp_image2)

ValueError: ExternalFile must specify at least one of 'file_content', 'file_name', 'file_pointer_meta' or 'file_descriptor_meta'.

In [ ]:
embedding_result

In [ ]:
ImageEmbedder.cosine_similarity(
  embedding_result.embeddings[0],
  embedding_result2.embeddings[0])

# Towhee test embedding

In [15]:
from towhee import AutoPipes

p = AutoPipes.pipeline('image-embedding')


In [16]:
res = p([img_path, img_path2])
res.get()

RuntimeError: Node-image-decode-0 runs failed, error msg: 'list' object has no attribute 'startswith', Traceback (most recent call last):
  File "/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/towhee/runtime/nodes/node.py", line 158, in _call
    return True, self._op(*inputs), None
  File "/home/superdave/.towhee/operators/towhee/image-decode/versions/main/image_decode.py", line 14, in __call__
    return self._op(image_path)
  File "/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/towhee/runtime/factory.py", line 125, in __call__
    result = self._op(*args, **kws)
  File "/home/superdave/.towhee/operators/image-decode/cv2/versions/main/image_decode_cv2.py", line 61, in __call__
    elif image_data.startswith('http'):
AttributeError: 'list' object has no attribute 'startswith'
, Traceback (most recent call last):
  File "/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/towhee/runtime/nodes/node.py", line 171, in process
    self

# Resnet from Chat GPT

In [ ]:
import numpy as np

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

def initialize_resnet_model(model_name='resnet50'):
    # Load the ResNet model
    if model_name == 'resnet18':
        model = models.resnet18(pretrained=True)
    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=True)
    else:
        raise ValueError("Unsupported model name. Supported options are 'resnet18' and 'resnet50'.")

    # Remove the final classification layer
    model = nn.Sequential(*list(model.children())[:-1])
    
    # Set the model to evaluation mode
    model.eval()
    
    return model

def load_and_preprocess_images(image_paths, target_size=(640, 640)):
    images = []

    for image_path in image_paths:
        # Load and preprocess each image
        img = Image.open(image_path)

        # Calculate padding if the image is smaller than the target size
        width, height = img.size
        if width < target_size[0] or height < target_size[1]:
            padding_width = max(target_size[0] - width, 0)
            padding_height = max(target_size[1] - height, 0)

            # Create a new image with padding
            padding = (0, 0, padding_width, padding_height)
            img = Image.new(img.mode, target_size, (255, 255, 255))
            img.paste(Image.open(image_path), padding)

        # Resize the image to the target size
        transform = transforms.Compose([
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        img = transform(img)
        images.append(img)

    return torch.stack(images)  # Convert the list of images to a single tensor

def extract_resnet_features(image_paths, batch_size=32, model_name='resnet50'):
    model = initialize_resnet_model(model_name)

    features_list = []
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i + batch_size]
        batch_images = load_and_preprocess_images(batch_paths)

        # Get the feature embeddings for the batch
        with torch.no_grad():
            features = model(batch_images)

        # Flatten the feature tensor for each image in the batch
        features = features.view(features.size(0), -1)

        # Append the features to the list
        features_list.extend(features.numpy())

    return np.vstack(features_list)

# Example usage:
image_paths = [img_path, img_path2]  # Add all your image paths here
feature_embeddings = extract_resnet_features(image_paths, batch_size=64)

# 'feature_embeddings' will contain the feature embeddings for all the images


In [ ]:
feature_embeddings

## data loader version

In [1]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import numpy as np
from torch.utils.data import DataLoader, Dataset

def initialize_resnet_model(model_name='resnet50'):
    # Load the ResNet model and move it to the GPU if available
    if model_name == 'resnet18':
        model = models.resnet18(pretrained=True)
    elif model_name == 'resnet50':
        model = models.resnet50(pretrained=True)
    else:
        raise ValueError("Unsupported model name. Supported options are 'resnet18' and 'resnet50'.")

    # Remove the final classification layer
    model = nn.Sequential(*list(model.children())[:-1])

    # Move the model to the GPU if available
    if torch.cuda.is_available():
        model = model.to('cuda')

    # Set the model to evaluation mode
    model.eval()

    return model

model = initialize_resnet_model()

/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:
class CustomDataset(Dataset):
    def __init__(self, image_paths, target_size=(640, 640)):
        self.image_paths = image_paths
        self.transform = transforms.Compose([
            transforms.Resize(target_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx])

        # Calculate padding if the image is smaller than the target size
        width, height = img.size
        if width < 640 or height < 640:
            padding_width = max(640 - width, 0)
            padding_height = max(640 - height, 0)

            # Create a new image with padding
            padding = (0, 0, padding_width, padding_height)
            img = Image.new(img.mode, (640, 640), (255, 255, 255))
            img.paste(Image.open(self.image_paths[idx]), padding)

        img = self.transform(img)

        # Move the image to the GPU if available
        if torch.cuda.is_available():
            img = img.to('cuda')

        return img

In [3]:
def extract_resnet_features(model, image_paths, batch_size=32):
    
    dataset = CustomDataset(image_paths)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    features_list = []

    with torch.no_grad():
        for batch in dataloader:

            features = model(batch)
            features = features.view(features.size(0), -1)
            # L2 normalize the features
            features /= features.norm(p=2, dim=1, keepdim=True)
            features_list.extend(features.detach().cpu().numpy())

    return np.vstack(features_list)

In [11]:
# Example usage:
image_path_base = [img_path, img_path2]  # Add all your image paths here
image_paths = [ele for ele in image_path_base for i in range(100)]
feature_embeddings = extract_resnet_features(model, image_paths, batch_size=64)

# 'feature_embeddings' will contain the feature embeddings for all the images
# 50 images 14 sec
# 100 images 23.6
# 200 images 45.5

In [8]:
feature_embeddings

array([[0.00603489, 0.0146739 , 0.01217721, ..., 0.00940299, 0.02002349,
        0.00984282],
       [0.00603489, 0.0146739 , 0.01217721, ..., 0.00940299, 0.02002349,
        0.00984282],
       [0.00603489, 0.0146739 , 0.01217721, ..., 0.00940299, 0.02002349,
        0.00984282],
       ...,
       [0.00771621, 0.01992542, 0.01226389, ..., 0.0046333 , 0.01500007,
        0.01592371],
       [0.00771621, 0.01992542, 0.01226389, ..., 0.0046333 , 0.01500007,
        0.01592371],
       [0.00771621, 0.01992542, 0.01226389, ..., 0.0046333 , 0.01500007,
        0.01592371]], dtype=float32)

In [9]:
len(feature_embeddings)

50

In [ ]:
47.6/200